In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd 
import pathlib as Path

In [ ]:
print(type("1,250"))
print(int("09790"))
print(float("58.9"))
print(0.1 + 0.2)
print(round(7.5), round(8.5))

In [ ]:
from olist_analysis import cleaning 

tests = ["1,250", " 1,250.50 ", "฿99", "120 บาท", 75, "", "ไม่ทราบ", None, float("nan"), True]

for t in tests:
    print(repr(t),'->',cleaning.clean_price(t))

In [ ]:
day_late = [-3, 0, 1, 7, 8, None]

for d in day_late:
    print(d, '-->', cleaning.delivery_status(d))



In [ ]:
required = ["order_id", "order_status", "order_delivered_customer_date", "order_estimated_delivery_date"]
have = ["order_id", "customer_id", "order_status", "order_purchase_timestamp"]

# ข้อ 1: for loop หาตัวที่ขาด
for r in required:
    if r not in have:
        print(f"{r} is missing")
# ข้อ 2: list comprehension บรรทัดเดียว 
not_have = [r for r in required if r not in have]
print(not_have)

In [ ]:
from olist_analysis import load
from pathlib import Path

required = ["order_id", "order_status", "order_delivered_customer_date", "order_estimated_delivery_date"]

order = load.read_table('orders', Path("../data/raw/olist/"))
cleaning.require_columns(order, required)
cleaning.require_columns(order, required + ['discount'])



In [ ]:
clean = cleaning.clean_orders(order)
clean[['delivery_days', 'days_late']].describe()

bins = [0,7,14,30,60,100,float('inf')]
labels = ['0-7','8-14','15-30','31-60','61-100','>100']

clean['days_bin'] = pd.cut(clean['delivery_days'], bins=bins, labels=labels)
clean['days_bin'].value_counts(sort=False)

In [ ]:
long = clean[clean['delivery_days'] > 100]

long.groupby(long['order_purchase_timestamp'].dt.to_period('M')).size()

In [ ]:
delivered = clean[clean['delivery_days'].notna()]
month = delivered['order_purchase_timestamp'].dt.to_period('M')
month.value_counts()

rate = (delivered['delivery_days']>100).groupby(month).mean()*100
round(rate, 3)

In [ ]:
print(order['order_purchase_timestamp'].max())
print(order['order_delivered_customer_date'].max())


In [ ]:
clean = cleaning.clean_orders(order)
clean['delivery_status'].value_counts()
clean['is_date_error'].sum()


In [ ]:
assert 1+1 == 2
assert 1+1 == 3

In [ ]:
with open("try_with.txt", "w") as f:
    f.write("hello")
    print("ข้างใน:", f.closed)

print("ข้างนอก:", f.closed)

In [ ]:
try:
    with open("try_with.txt") as f:
        print("ข้างใน:", f.closed)
        1 / 0
except ZeroDivisionError:
    print("เกิด error แล้ว")

print("ข้างนอก:", f.closed)

In [ ]:
Path.cwd()

In [ ]:
def count_missing(df: pd.DataFrame, *columns:str) -> dict[str, int] :
    return {c: int(df[c].isna().sum()) for c in columns}
        


count_missing(order, "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date")

In [ ]:
raw_dir =Path('../data/raw/olist')

for name in load.RAW_FILES.values():
    print(name)


In [ ]:
small = load.read_table("orders", Path("../data/raw/olist"))

len(small)


In [ ]:
def summarize_by(df: pd.DataFrame, *cols: str, 
        value: str='delivery_days', min_n: int=30) -> pd.DataFrame:
    '''สรุปจำนวน และมัธยะฐานแยกตาม columns ที่ระบุตัดกลุ่มที่มีจำนวนเล็กกว่า 30'''
    out = (df.groupby(list(cols), observed=True)[value]
        .agg(n='size', n_notna='count', median='median')
        .reset_index())
    return out[out['n_notna'] >= min_n]

order = cleaning.clean_orders(order)
order_st = order_status = cleaning.clean_orders(order)
order_st['purchase_month'] = order['order_purchase_timestamp'].dt.to_period('M')

display(summarize_by(order_st, "delivery_status"))
display(summarize_by(order_st, "purchase_month", "delivery_status"))
display(summarize_by(order_st, "delivery_status", value="days_late"))


In [ ]:
class CountingClicker:
    """A class can/should have a docstring, just like a function"""

    def __init__(self, count=0):
        self.count = count

    def click(self, num_times=1):
        """Click the clicker some number of times."""
        self.count += num_times

    def read(self):
        return self.count

    def reset(self):
        self.count = 0


font = CountingClicker()
back = CountingClicker()

font.click()
font.click()
font.click(5)
back.click()

print(font.read(), back.read())
back.reset()
print(font.read(), back.read())

In [ ]:
from olist_analysis.config import ReportConfig
from olist_analysis import metrics

cfg = ReportConfig(raw_dir=Path("../data/raw/olist"))   
order = load.read_table("orders", cfg.raw_dir)
clean = cleaning.clean_orders(order)
metrics.summarize_by(clean, "delivery_status", min_n=cfg.min_n)

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class CheckResult:
    """ผลตรวจหนึ่งข้อ (แบบฟอร์มที่ผู้จัดการจด)"""
    name: str
    n_bad: int
    passed: bool


class Check:
    """แม่พิมพ์ของตัวตรวจทุกแบบ"""
    name = "check"

    def bad_rows(self, df: pd.DataFrame) -> pd.Series:
        raise NotImplementedError("ตัวตรวจลูกต้องเขียน bad_rows เอง")

    def run(self, df: pd.DataFrame) -> CheckResult:
        n_bad = int(self.bad_rows(df).sum())
        return CheckResult(name=self.name, n_bad=n_bad, passed=(n_bad == 0))


class NotNull(Check):
    """แถวที่คอลัมน์นี้ว่าง ถือว่าเสีย"""

    def __init__(self, column: str):
        self.column = column
        self.name = f"not_null({column})"

    def bad_rows(self, df: pd.DataFrame) -> pd.Series:
        return df[self.column].isna()


class NotNegative(Check):
    def __init__(self, column: str):
        self.column = column
        self.name = f"not_negative({column})"

    def bad_rows(self, df: pd.DataFrame) -> pd.Series:
        return df[self.column] < 0



In [ ]:
test =[
    NotNull("order_id"),
    NotNull("order_delivered_customer_date"),
    NotNegative("delivery_days")
]

pd.DataFrame([t.run(clean) for t in test])


In [ ]:
clean['status_group'] = clean['order_status'].map(cleaning.status_group)

clean['status_group'].value_counts()

In [ ]:
from olist_analysis import load, config

@dataclass(frozen=True)
class CheckResult:
    name : str
    n_bad : int
    passed : bool


class Check:
    name = 'check'

    def bad_row(self, column):
        raise NotImplementedError('ไม่มีฟังชั่นนี้อยู่ในคลาสลูก')

    def run(self, df):
        n_bad = int(self.bad_row(df).sum())
        return CheckResult(name = self.name, n_bad = n_bad, passed = (n_bad == 0))


class InRange(Check):
    def __init__(self, column, low, hight):
         self.column = column
         self.low = low
         self.hight = hight
         self.name = f'in_rage ({column})'

    def bad_row(self, df):
        col = df[self.column]
        good = col.between(self.low, self.hight)
        return ~good

class NotBlank(Check):
    def __init__(self, column):
        self.column = column
        self.name = f'not_blank ({column})'

    def bad_row(self, df):
        col = df[self.column]
        return col.str.strip() == ""

class HasLenght(Check):
    def __init__(self, column, lenght):
        self.column = column
        self.lenght = lenght
        self.name = f'has_lenght'

    def bad_row(self, df):
        return df[self.column].str.len() != self.lenght


review = pd.read_csv('/Users/ohm/projects/olist-analysis/data/raw/olist/olist_order_reviews_dataset.csv')
customer = pd.read_csv('/Users/ohm/projects/olist-analysis/data/raw/olist/olist_customers_dataset.csv')
customer2 = load.read_table('customer', config.ReportConfig.raw_dir)





In [ ]:
from olist_analysis import checks

order_item = load.read_table('order_items', config.ReportConfig.raw_dir)

ei = checks.ExistsIn('order_id', clean, 'order_id')
ei.bad_row(order_item)
ei.run(order_item)
